---
title: "Part 4b: NumPy Advanced Operations"
---


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/04b-numpy.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/04b-numpy.ipynb)

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

Part 4a gave you the fundamentals: creating arrays, checking shapes, indexing, masking, and aggregating. That is enough to write correct NumPy code.

Broadcasting and vectorisation are what make NumPy fast. A subtraction like <code>X - mean_row</code> that looks like a shape mismatch is actually valid -- NumPy stretches the smaller array to match. Understanding this rule is what separates code that works from code that works quickly.

This notebook covers broadcasting, vectorisation, linear algebra, saving/loading, and gotchas. By the end you will have everything you need for Part 5: matplotlib visualisations of the arrays you have been building.

> Callout markers: [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Before You Begin

Run Part 4a (`04a-numpy.ipynb`) first. The Setup section below re-imports NumPy and rebuilds the feature matrix used throughout Part 4a.
:::

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

| # | Skill | Covered in |
|---|---|---|
| 1 | Apply broadcasting rules to combine arrays of different shapes | Sec. 7 |
| 2 | Replace Python loops with vectorised NumPy expressions | Sec. 8 |
| 3 | Compute dot products and matrix operations with `@` | Sec. 9 |
| 4 | Save and load arrays with `.npy` and `.npz` | Sec. 10 |
| 5 | Identify and fix the three most common NumPy gotchas | Sec. 11 |
:::

## Setup

Re-run this cell to restore the imports and feature matrix from Part 4a.

In [ ]:
import numpy as np

# Feature matrix from Part 4a: 5 students x 3 features
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
        [22.0, 98.0, 3.9],
    ]
)
feature_names = ["study_hours", "attendance_pct", "prior_gpa"]
print(f"X.shape={X.shape}  X.dtype={X.dtype}")

## 7. Broadcasting

**Broadcasting** is the rule NumPy uses to apply an operation between two arrays of *different* shapes, by virtually "stretching" the smaller one, without actually copying any data. It is what lets you write `X - X.mean(axis=0)` instead of a loop over rows.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: The Broadcasting Rule</span><br><br>
Compare shapes from the <b>right-hand side</b>. Two dimensions are compatible when they are <b>equal</b>, or when <b>one of them is 1</b> (it gets stretched to match). Missing leading dimensions are treated as 1.<br><br> <code>(5, 3)</code> and <code>(3,)</code> &rarr; treat <code>(3,)</code> as <code>(1, 3)</code> &rarr; stretch to <code>(5, 3)</code>. ✅<br> <code>(5, 3)</code> and <code>(5,)</code> &rarr; treat <code>(5,)</code> as <code>(1, 5)</code> &rarr; <code>3 != 5</code> and neither is 1. ❌
</div>

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Broadcasting: shapes align from the right, size-1 dimensions stretch</span><br><br>
NumPy lines up shapes from the right. A dimension of size 1 is stretched to match the other array. Everything else must match exactly -- otherwise NumPy raises a <code>ValueError</code>. The rule: <code>(5, 3)</code> minus <code>(3,)</code> is valid; NumPy treats the <code>(3,)</code> as <code>(1, 3)</code> and stretches to <code>(5, 3)</code>.
</div>

In [ ]:
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
        [22.0, 98.0, 3.9],
    ]
)

feature_mean = X.mean(axis=0)  # shape (3,)  -- one mean per feature
feature_std = X.std(axis=0)  # shape (3,)

print(f"X.shape            : {X.shape}")
print(f"feature_mean.shape : {feature_mean.shape}")

# (5, 3) - (3,) broadcasts the mean across every row: no loop needed
X_normalised = (X - feature_mean) / feature_std
print(f"normalised:\n{X_normalised}")
print(f"new per-feature mean (~0): {X_normalised.mean(axis=0).round(6)}")

The rule is: align shapes from the right. Dimensions of size 1 stretch to match. Everything else must match exactly or NumPy raises an error.

![NumPy broadcasting: a (5,3) matrix plus a (3,) vector. The vector is stretched to (5,3) by repeating it across rows, producing a (5,3) result.](figs/broadcasting.svg){fig-alt="Left: blue 5x3 grid. Center: plus sign. Right: green 1x3 vector with ghost rows showing it stretches to 5x3. Arrow to purple 5x3 result grid."}

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Broadcasting a (5,) array against a (5, 3) matrix fails for a subtle reason</span><br><br>
If you instead compute <code>per_student_mean = X.mean(axis=1)</code> (shape <code>(5,)</code>) and try <code>X - per_student_mean</code>, NumPy raises <code>ValueError: operands could not be broadcast together</code>. It is comparing the trailing dimensions <code>3</code> vs <code>5</code>, not what you meant. Fix it by giving the per-row result an explicit column shape with <code>keepdims=True</code>: <code>X.mean(axis=1, keepdims=True)</code> has shape <code>(5, 1)</code>, which broadcasts correctly against <code>(5, 3)</code>.
</div>

In [ ]:
row_mean = X.mean(axis=1, keepdims=True)  # shape (5, 1), NOT (5,)
print(f"row_mean.shape : {row_mean.shape}")

# (5, 3) - (5, 1) broadcasts the per-row mean across every column
centered_per_row = X - row_mean
print(f"centered_per_row:\n{centered_per_row}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Min-Max Scale Every Feature</span><br><br>

<b>Goal:</b> Write a one-line expression that scales every column of <code>X</code> to the <code>[0, 1]</code> range using the formula <code>(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))</code>. Confirm the result's per-column min is 0 and max is 1.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>X_scaled.min(axis=0)  # -> array([0., 0., 0.])
X_scaled.max(axis=0)  # -> array([1., 1., 1.])</pre>
</div>

In [ ]:
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
        [22.0, 98.0, 3.9],
    ]
)

X_scaled = ...  # TODO: min-max scale every column to [0, 1]

# print(f"min per column: {X_scaled.min(axis=0)}")
# print(f"max per column: {X_scaled.max(axis=0)}")

## 8. Vectorisation vs. Python Loops

Now that you can express normalisation as `(X - mean) / std` with no loop, it is worth seeing *why* that matters. Every NumPy operation you have used so far runs as a single compiled loop over contiguous memory; a Python `for` loop pays the cost of the interpreter on every single element.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Replace loops with vectorised expressions: 10-100x faster</span><br><br>
A Python loop over one million floats takes ~500ms. The same operation as <code>(arr - arr.mean()) / arr.std()</code> takes ~5ms. NumPy calls compiled C code -- the Python interpreter never enters the inner loop. The speedup grows with array size.
</div>

In [ ]:
import time

big = np.random.default_rng(0).normal(size=1_000_000)


def zscore_loop(values: np.ndarray) -> list[float]:
    mean = sum(values) / len(values)
    variance = sum((v - mean) ** 2 for v in values) / len(values)
    std = variance**0.5
    return [(v - mean) / std for v in values]


start = time.perf_counter()
_ = zscore_loop(big)
loop_time = time.perf_counter() - start

start = time.perf_counter()
_ = (big - big.mean()) / big.std()
vector_time = time.perf_counter() - start

print(f"Python loop time : {loop_time:.4f}s")
print(f"Vectorised time  : {vector_time:.4f}s")
print(f"Speedup          : {loop_time / vector_time:,.0f}x")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: If you are writing a <code>for</code> loop over an array, stop and look for a vectorised way</span><br><br>
Almost every elementwise transformation, filter, or aggregation you would write as a Python loop already has a NumPy equivalent: arithmetic operators, <code>np.where</code>, boolean masks, <code>axis</code> aggregations. Reach for those first: a hand-written loop over a large array is one of the most common DS/ML performance bugs, and it is usually a 10-100x slowdown for no benefit.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Benchmark: loop vs vectorised z-score</span><br><br>
<b>Goal:</b> Measure the speedup of vectorisation.<br><br>1. Create <code>arr = np.random.default_rng(0).normal(size=100_000)</code>.<br>2. Implement a Python loop z-score (iterate element by element).<br>3. Implement the vectorised version: <code>(arr - arr.mean()) / arr.std()</code>.<br>4. Time both with <code>import time</code> and print the speedup ratio.<br><br><b>Expected:</b> vectorised is 50-200x faster.
</div>

## 9. Linear Algebra Essentials

A linear model's prediction is a **dot product**: multiply each feature by a learned weight, sum the results, add a bias. `@` (matrix multiplication) computes this for every row of a feature matrix at once, no loop over students required.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: @ is matrix multiplication -- use it, never loop</span><br><br>
<code>X @ w</code> computes every row's dot product with <code>w</code> in one call. Manually looping <code>for row in X: sum(row * w)</code> is correct but 100x slower. Every linear model, transformer attention layer, and PCA uses <code>@</code> internally.
</div>

In [ ]:
X = np.array(
    [
        [12.0, 85.0, 3.1],
        [5.0, 60.0, 2.4],
        [18.0, 95.0, 3.8],
        [9.0, 70.0, 2.9],
        [22.0, 98.0, 3.9],
    ]
)  # shape (5 students, 3 features)

# Suppose a (already-fitted) linear model has these learned weights and bias
weights = np.array([1.5, 0.3, 8.0])  # one weight per feature, shape (3,)
bias = 10.0

# X @ weights: (5, 3) @ (3,) -> (5,) -- one prediction per student
predicted_scores = X @ weights + bias
print(f"predicted_scores: {predicted_scores.round(1)}")

`@` on a matrix and a vector is shorthand for: for each row, multiply element-wise by `weights` and sum: exactly `(X * weights).sum(axis=1)`. Verify the two are equivalent, then measure prediction error against the true scores with `np.linalg.norm` (the Euclidean / RMS-style distance):

In [ ]:
# @ is equivalent to elementwise multiply + sum along axis=1
manual = (X * weights).sum(axis=1) + bias
print(f"@ matches manual sum: {np.allclose(predicted_scores, manual)}")

actual_scores = np.array([88.0, 65.0, 95.0, 78.0, 99.0])
errors = predicted_scores - actual_scores
rmse = np.linalg.norm(errors) / np.sqrt(len(errors))
print(f"errors : {errors.round(1)}")
print(f"RMSE   : {rmse:.2f}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Comparing floats with <code>==</code></span><br><br>
<code>np.allclose(a, b)</code> was used above instead of <code>(a == b).all()</code> on purpose: floating-point arithmetic accumulates tiny rounding errors, so two mathematically-equal results can differ in their last bit. Always compare floats with a tolerance: <code>np.allclose</code>, or <code>abs(a - b) &lt; 1e-9</code>, never with exact <code>==</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Linear prediction with @</span><br><br>
<b>Goal:</b> Use matrix multiplication to compute predictions.<br><br>Using <code>X</code> from the setup section:<br>1. Define <code>weights = np.array([0.3, 0.4, 1.5])</code> and <code>bias = -2.0</code>.<br>2. Compute predicted scores as <code>preds = X @ weights + bias</code>.<br>3. Confirm <code>preds.shape == (5,)</code>.<br>4. Print the index of the highest-scoring student using <code>np.argmax(preds)</code>.
</div>

## 10. Saving & Loading Arrays

A typical pipeline computes a feature matrix once and reuses it across many later steps (training, evaluation, serving). `.npy` stores a single array in NumPy's own binary format: far smaller and faster to read than CSV for numeric data, and it preserves `dtype` and `shape` exactly. `.npz` bundles several named arrays together.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Use .npy for one array, .npz for multiple</span><br><br>
<code>np.save('X.npy', X)</code> and <code>np.load('X.npy')</code> preserve dtype and shape exactly. <code>np.savez('data.npz', X=X, y=y)</code> bundles multiple arrays in one file. Both load in microseconds -- much faster than re-parsing a CSV.
</div>

In [ ]:
from pathlib import Path

tmp_dir = Path("tmp_numpy_activity")
tmp_dir.mkdir(exist_ok=True)

X = np.array([[12.0, 85.0, 3.1], [5.0, 60.0, 2.4], [18.0, 95.0, 3.8]])
y = np.array([88.0, 65.0, 95.0])

# Save a single array
np.save(tmp_dir / "X.npy", X)
X_loaded = np.load(tmp_dir / "X.npy")
print(f"round-trip equal: {np.array_equal(X, X_loaded)}")

# Save several named arrays together in one file
np.savez(tmp_dir / "dataset.npz", features=X, target=y)
bundle = np.load(tmp_dir / "dataset.npz")
print(f"keys   : {list(bundle.keys())}")
print(f"target : {bundle['target']}")

In [ ]:
import shutil

shutil.rmtree(tmp_dir)
print(f"cleaned up: {tmp_dir.exists()}")

## 11. Common Gotchas

Like the Python gotchas in Part 3, none of these raise an exception. They silently produce a wrong (or surprising) result. Recognise them now so you do not lose hours to them later.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Three gotchas that never raise an exception</span><br><br>
1. <b>View vs copy</b>: <code>sub = X[:2]</code> shares memory -- mutating <code>sub</code> mutates <code>X</code>. Fix: <code>X[:2].copy()</code>.<br>2. <b>Integer overflow</b>: <code>np.array([120, 10], dtype=np.int8)</code> wraps silently. Fix: check dtype before arithmetic.<br>3. <b>Shape mismatch in broadcasting</b>: <code>(5, 3) - (3, 5)</code> may broadcast unexpectedly. Always check shapes before subtracting.
</div>

In [ ]:
# GOTCHA 1: basic slicing returns a VIEW, not a copy
scores = np.array([62, 78, 85, 91, 55])
top_three = scores[:3]
top_three[0] = 0  # mutating the "slice"...

print(f"top_three : {top_three}")
print(f"scores    : {scores}")  # ...changed the original too!

# Fix: copy explicitly when you need an independent array
safe_copy = scores[:3].copy()

In [ ]:
# GOTCHA 2: integer dtype truncates on division-like ops, and can overflow
small = np.array([120, 10], dtype=np.int8)
print(f"int8 + 50 : {small + 50}")  # 120 + 50 = 170, overflows int8's max of 127!

# Fix: use a wide-enough dtype, or let NumPy infer (default is int64/float64)
safe = small.astype(np.int64) + 50
print(f"int64 + 50: {safe}")

In [ ]:
# GOTCHA 3: {} is a dict, not a set -- same trap as in plain Python (Part 3, Sec. 8)
empty_dict = {}
empty_set = set()
print(f"type({{}})   : {type(empty_dict)}")
print(f"type(set()) : {type(empty_set)}")

# GOTCHA 4: comparing floats with == (see Sec. 9 for the fix: np.allclose)
a = np.array([0.1 + 0.2])
print(f"0.1 + 0.2 == 0.3 : {a == 0.3}")  # False! 0.30000000000000004 != 0.3
print(f"np.allclose      : {np.allclose(a, 0.3)}")  # True: tolerant comparison

## 12. Capstone Exercises

Apply everything from this notebook together. Each exercise is self-contained.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Vectorised Anomaly Detector</span><br><br>

<b>Goal:</b> Rewrite the deque-based anomaly detector from Part 3 (Sec. 9, Exercise 3) without any explicit loop. Flag any reading more than <code>2</code> standard deviations from the <b>overall</b> mean using a single boolean mask.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>readings = [36.5, 36.7, 36.8, 36.6, 36.9, 39.5, 36.7, 36.8]
# Expected: reading 39.5 (index 5) flagged as anomaly</pre>
<b>Hint:</b> <code>z = (readings - readings.mean()) / readings.std()</code>, then mask <code>np.abs(z) > 2</code>.
</div>

In [ ]:
readings = np.array([36.5, 36.7, 36.8, 36.6, 36.9, 39.5, 36.7, 36.8])

z_scores = ...  # TODO
anomaly_mask = ...  # TODO

if z_scores is not ...:
    print(f"z_scores      : {z_scores.round(2)}")
    print(f"anomaly_mask  : {anomaly_mask}")
    print(f"anomalies     : {readings[anomaly_mask]}")
else:
    print("(implement z_scores and anomaly_mask above to see output)")

## What's New in NumPy 2.0

NumPy 2.0 (released June 2024) is the first major version bump in almost two decades. Two changes matter most for day-to-day data science work:

### Removed type aliases

The old Python-builtin aliases (`np.int`, `np.float`, `np.bool`, `np.complex`, `np.object`, `np.str`) were deprecated for years and are now **fully removed**. They were just aliases to the Python built-ins anyway, so the fix is mechanical:

| Old (removed) | Replacement |
|---|---|
| `np.bool` | `np.bool_` or Python `bool` |
| `np.int` | `np.intp` or Python `int` |
| `np.float` | `np.float64` or Python `float` |
| `np.complex` | `np.complex128` or Python `complex` |
| `np.object` | `np.object_` or Python `object` |
| `np.str` | `np.str_` or Python `str` |

### `StringDType` for proper string arrays

NumPy 2.0 introduced `np.dtypes.StringDType()`, a real variable-length string dtype backed by UTF-8 memory. The old `np.str_` stored fixed-width UCS-4 strings (one array dtype for every character count), `StringDType` stores arbitrary-length strings efficiently.

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>np.float64</code> not <code>np.float</code> in new code</span><br><br>
If you see a <code>module 'numpy' has no attribute 'float'</code> error, the codebase was written against NumPy 1.x. A global search-and-replace of <code>np.float</code> → <code>np.float64</code> (and so on for the others in the table) is the complete fix.
</div>

In [ ]:
import numpy as np

# Old aliases are REMOVED in NumPy 2.0: this would raise AttributeError:
# arr = np.array([1, 2, 3], dtype=np.float)   # ❌

# Use the explicit dtype names instead:
arr = np.array([1, 2, 3], dtype=np.float64)  # ✅
print(f"dtype: {arr.dtype}")

# StringDType: variable-length strings, efficient UTF-8 storage (NumPy 2.0+)
names = np.array(["Alice", "Bob", "Charlie"], dtype=np.dtypes.StringDType())
print(f"names : {names}")
print(f"dtype : {names.dtype}")

# The old fixed-width str_ is still available but StringDType is the modern choice
old_style = np.array(["Alice", "Bob", "Charlie"])  # infers str_ (fixed width)
print(f"old dtype: {old_style.dtype}")  # <U7 means Unicode, 7 chars wide

### `np.strings`: Vectorised String Operations

NumPy 2.0 ships a dedicated `np.strings` module for vectorised string operations on `np.ndarray` values backed by `StringDType`. Before this module existed, string operations on object arrays required Python-level loops or `np.vectorize`, both of which are slow. `np.strings` routes each operation through a compiled C loop, making it roughly 10x faster on large arrays than the equivalent Python loop.

The module mirrors the Python `str` method set: `np.strings.upper()`, `np.strings.lower()`, `np.strings.split()`, `np.strings.startswith()`, `np.strings.replace()`, and others all work element-wise, broadcasting across the entire array in one call.

In [ ]:
import numpy as np

# Create a StringDType array of course codes
course_arr = np.array(["DATA101", "MATH201", "STATS301", "ML401"], dtype=np.dtypes.StringDType())

# np.strings.upper() broadcasts across the whole array in one compiled call
upper = np.strings.upper(course_arr)
print("upper:", upper)

# np.strings.startswith() returns a boolean array, usable as a mask
prefix_mask = np.strings.startswith(course_arr, "DATA")
print("DATA prefix mask:", prefix_mask)
print("courses starting with 'DATA':", course_arr[prefix_mask])

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>np.dtypes.StringDType()</code> for string arrays that need fast operations</span><br><br>
The default <code>np.array(["a", "b"])</code> stores strings as <code>object</code> dtype: each element is a Python object pointer, so any operation loops at the Python level. <code>np.dtypes.StringDType()</code> stores strings as contiguous UTF-8 bytes, avoids the object-dtype overhead, and unlocks the entire <code>np.strings.*</code> API for compiled, vectorised string work.
</div>

## Further Reading

| Resource | Why it matters |
|---|---|
| Harris, C.R. et al. (2020). [Array programming with NumPy](https://doi.org/10.1038/s41586-020-2649-2). *Nature* 585, 357–362. | The primary citation for NumPy; the paper explains the design decisions behind broadcasting and ufuncs |
| VanderPlas, J. (2016). *Python Data Science Handbook*, Ch. 2. O'Reilly. | Free online: the most readable treatment of fancy indexing, broadcasting, and structured arrays |
| [NumPy documentation: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) | Official broadcasting rules with diagrams; bookmark for the next time the shapes don't align |
| [NumPy documentation: Indexing](https://numpy.org/doc/stable/user/basics.indexing.html) | Covers basic, advanced, and boolean indexing in one place |


## Summary

| Concept | Key rule |
|---|---|
| `ndarray` | One dtype, contiguous memory, fast because it skips the Python interpreter |
| Creation | `np.array`, `arange`, `linspace`, `zeros`/`ones`; `np.random.default_rng(seed)` for reproducible random data |
| `shape`/`dtype` | Always check before trusting a result; mixed-type input silently upcasts |
| `reshape` | Returns a **view**, same data, same `size`, different shape |
| `flatten` | Always returns a **copy** |
| Slicing | Basic slices (`X[:, 0]`) are views; fancy/boolean indexing always copies |
| Boolean masks | `&` / `\|` (not `and`/`or`) on arrays, each side parenthesised |
| `np.where` / `np.select` | Vectorised if/else and multi-branch labelling |
| `axis=0` vs `axis=1` | 0 collapses rows -> one value per **column**; 1 collapses columns -> one value per **row** |
| Broadcasting | Compare shapes right-to-left; dims match if equal or one is `1`; use `keepdims=True` to broadcast a per-row stat |
| Vectorisation | A NumPy expression beats a Python loop by 10-100x, look for one before writing `for` |
| `@` / `np.dot` | Matrix multiplication: `X @ weights` predicts every row in one call |
| `np.allclose` | Always compare floats with a tolerance, never `==` |
| `.npy` / `.npz` | Compact, dtype/shape-preserving array storage between pipeline stages |


**Next:** `05-matplotlib.ipynb`, covering how to visualise arrays and DataFrames with matplotlib.